In [2]:
"""
data_cleaning.py
================
Data Cleaning Pipeline for Financial Sentiment Analysis Dataset
"""

import re
import logging
import pandas as pd
import numpy as np

# ── Logging Setup ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)


# ── Constants ─────────────────────────────────────────────────────────────────
INPUT_PATH  = "data.csv"
OUTPUT_PATH = "data_cleaned.csv"

LABEL_MAP = {
    "positive": "positive",
    "negative": "negative",
    "neutral":  "neutral"
}


# ─────────────────────────────────────────────────────────────────────────────
# Step 1 — Load Data
# ─────────────────────────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, header=0)
    df.columns = ["Sentence", "Sentiment"]
    logger.info(f"Loaded  →  {df.shape[0]:,} rows  |  {df.shape[1]} columns")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 2 — Drop Nulls
# ─────────────────────────────────────────────────────────────────────────────
def drop_nulls(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.dropna(subset=["Sentence", "Sentiment"])
    dropped = before - len(df)
    logger.info(f"Null rows dropped  →  {dropped}")
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# Step 3 — Remove Duplicates
# ─────────────────────────────────────────────────────────────────────────────
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates(subset=["Sentence"]).reset_index(drop=True)
    dropped = before - len(df)
    logger.info(f"Duplicate rows removed  →  {dropped}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 4 — Normalize Labels
# ─────────────────────────────────────────────────────────────────────────────
def normalize_labels(df: pd.DataFrame) -> pd.DataFrame:
    df["Sentiment"] = df["Sentiment"].str.strip().str.lower()
    before = len(df)
    df = df[df["Sentiment"].isin(LABEL_MAP.keys())].reset_index(drop=True)
    dropped = before - len(df)
    logger.info(f"Invalid label rows removed  →  {dropped}")
    logger.info(f"Label distribution:\n{df['Sentiment'].value_counts().to_string()}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 5 — Clean Text
# ─────────────────────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    # Lowercase
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove stock tickers like $AAPL, $ESI
    text = re.sub(r"\$[A-Z]{1,5}", " ", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove numbers (keep if part of a word, e.g. "q3")
    text = re.sub(r"\b\d+\.?\d*\b", " ", text)

    # Remove special characters — keep only letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def apply_text_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    df["Cleaned_Sentence"] = df["Sentence"].apply(clean_text)
    logger.info("Text cleaning applied  →  'Cleaned_Sentence' column added")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 6 — Remove Empty Rows After Cleaning
# ─────────────────────────────────────────────────────────────────────────────
def drop_empty_cleaned(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df[df["Cleaned_Sentence"].str.strip().str.len() > 0].reset_index(drop=True)
    dropped = before - len(df)
    logger.info(f"Empty cleaned-text rows removed  →  {dropped}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 7 — Remove Very Short Sentences
# ─────────────────────────────────────────────────────────────────────────────
def drop_short_sentences(df: pd.DataFrame, min_words: int = 3) -> pd.DataFrame:
    before = len(df)
    df = df[df["Cleaned_Sentence"].apply(lambda x: len(x.split()) >= min_words)]
    dropped = before - len(df)
    logger.info(f"Short sentence rows removed (< {min_words} words)  →  {dropped}")
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# Step 8 — Add Feature Columns
# ─────────────────────────────────────────────────────────────────────────────
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df["word_count"]    = df["Cleaned_Sentence"].apply(lambda x: len(x.split()))
    df["char_count"]    = df["Cleaned_Sentence"].apply(len)
    df["label_encoded"] = df["Sentiment"].map({"positive": 2, "neutral": 1, "negative": 0})
    logger.info("Feature columns added  →  word_count, char_count, label_encoded")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Step 9 — Save Cleaned Data
# ─────────────────────────────────────────────────────────────────────────────
def save_data(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)
    logger.info(f"Cleaned data saved  →  {path}  ({len(df):,} rows)")


# ─────────────────────────────────────────────────────────────────────────────
# Summary Report
# ─────────────────────────────────────────────────────────────────────────────
def print_summary(original: pd.DataFrame, cleaned: pd.DataFrame) -> None:
    print("\n" + "="*55)
    print("           DATA CLEANING SUMMARY REPORT")
    print("="*55)
    print(f"  Original rows         : {len(original):,}")
    print(f"  Cleaned rows          : {len(cleaned):,}")
    print(f"  Rows removed          : {len(original) - len(cleaned):,}")
    print(f"  Retention rate        : {len(cleaned)/len(original)*100:.2f}%")
    print("-"*55)
    print("  Final Label Distribution:")
    for label, count in cleaned["Sentiment"].value_counts().items():
        pct = count / len(cleaned) * 100
        print(f"    {label:<12}  {count:>5,}  ({pct:.1f}%)")
    print("-"*55)
    print(f"  Avg word count        : {cleaned['word_count'].mean():.1f}")
    print(f"  Avg char count        : {cleaned['char_count'].mean():.1f}")
    print(f"  Min word count        : {cleaned['word_count'].min()}")
    print(f"  Max word count        : {cleaned['word_count'].max()}")
    print("="*55 + "\n")


# ─────────────────────────────────────────────────────────────────────────────
# Main Pipeline
# ─────────────────────────────────────────────────────────────────────────────
def run_pipeline():
    logger.info("Starting data cleaning pipeline...")

    df_raw = load_data(INPUT_PATH)
    df     = drop_nulls(df_raw.copy())
    df     = remove_duplicates(df)
    df     = normalize_labels(df)
    df     = apply_text_cleaning(df)
    df     = drop_empty_cleaned(df)
    df     = drop_short_sentences(df, min_words=3)
    df     = add_features(df)

    save_data(df, OUTPUT_PATH)
    print_summary(df_raw, df)

    return df


if __name__ == "__main__":
    cleaned_df = run_pipeline()

2026-04-11 22:59:29,043 | INFO | Starting data cleaning pipeline...


2026-04-11 22:59:29,106 | INFO | Loaded  →  5,842 rows  |  2 columns
2026-04-11 22:59:29,112 | INFO | Null rows dropped  →  0
2026-04-11 22:59:29,116 | INFO | Duplicate rows removed  →  520
2026-04-11 22:59:29,127 | INFO | Invalid label rows removed  →  0
2026-04-11 22:59:29,132 | INFO | Label distribution:
Sentiment
neutral     2878
positive    1852
negative     592
2026-04-11 22:59:29,690 | INFO | Text cleaning applied  →  'Cleaned_Sentence' column added
2026-04-11 22:59:29,707 | INFO | Empty cleaned-text rows removed  →  0
2026-04-11 22:59:29,724 | INFO | Short sentence rows removed (< 3 words)  →  15
2026-04-11 22:59:29,759 | INFO | Feature columns added  →  word_count, char_count, label_encoded
2026-04-11 22:59:29,893 | INFO | Cleaned data saved  →  data_cleaned.csv  (5,307 rows)



           DATA CLEANING SUMMARY REPORT
  Original rows         : 5,842
  Cleaned rows          : 5,307
  Rows removed          : 535
  Retention rate        : 90.84%
-------------------------------------------------------
  Final Label Distribution:
    neutral       2,874  (54.2%)
    positive      1,843  (34.7%)
    negative        590  (11.1%)
-------------------------------------------------------
  Avg word count        : 17.9
  Avg char count        : 104.8
  Min word count        : 3
  Max word count        : 50

